# harness_runs

> Harness configuration, runs and dynamic-range checks (Phase 7).

Spec: `specs/multiagent/phase7/`.

In [ ]:
#| default_exp harness_runs

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from dataclasses import asdict, dataclass, field
from typing import Optional

import numpy as np

from explRL.circuit import goalStats
from explRL.core import loadCircuitAgent
from explRL.harness import Blackboard, Lead, MessageBus
from explRL.harness_agents import CircuitCommMember, CommBandit, LeadFollower
from explRL.multi import runTeam, spawnRngs
from explRL.patchworld import TeamPatchWorld, ringLayout, torusLayout
from explRL.team_agents import randomAgent
from explRL.team_refs import (coverage, dangerShare, gini, idealFree, ifdDistanceTies, ledgerArrays, occupancy,
                        socialOptimum, timeToIFD, welfare, welfareLoss)


@dataclass
class HarnessConfig:
    channel: str = 'none'
    agent: str = 'bandit'
    w: float = 1.0
    K: Optional[int] = None
    lam: float = 0.01
    k: int = 2
    period: float = 10.0
    radius: float = np.inf
    budget: str = 'time'
    B: float = 15360.0
    T: float = 600.0
    cUCB: float = 1.0
    H: float = 10.0
    world: dict = field(default_factory=dict)

    def asDict(self):
        d = asdict(self)
        r = d['radius']
        if r == 'inf' or (isinstance(r, float) and not np.isfinite(r)):
            d['radius'] = 'inf'
        else:
            d['radius'] = float(r)
        return d

    @classmethod
    def fromDict(cls, d):
        r = d.get('radius', np.inf)
        if r == 'inf':
            r = np.inf
        return cls(**{**d, 'radius': r, 'world': dict(d.get('world', {}))})


def benchWorld(**overrides):
    L, travel, shape = overrides.pop('L', 64), overrides.pop('travelScale', 0.48), overrides.pop('shape', 'ring')
    layout = ringLayout(L, travel) if shape == 'ring' else torusLayout(int(round(L ** 0.5)), travel)   # plain values
    kw = dict(layout=layout, nPatches=16, nDanger=8, model='fixed',
              intake=(0.8, 1.6, 2.4, 3.2) * 4, Tmove=40.0, noise=0.45, dangerCost=2.6, c=0.85)
    kw.update(overrides)
    return TeamPatchWorld(**kw)


def circuitWorld(**overrides):
    return TeamPatchWorld.zoneConfig(symmetric=False, intake=1.0, **overrides)


def worldTime(N, cfg):
    if N < 1:
        raise ValueError('team needs at least one agent')
    v = cfg.B / N if cfg.budget == 'compute' else cfg.T
    if not v > 0:
        raise ValueError('budget must be positive')
    return float(v)


def makeChannel(cfg, L, N, seed):
    if cfg.channel == 'none' or cfg.agent == 'random':
        return None
    if cfg.channel == 'blackboard':
        return Blackboard(L, cfg.K, cfg.lam, N)
    if cfg.channel == 'messages':
        return MessageBus(N, cfg.k, cfg.period, cfg.radius, spawnRngs(seed + 2000, N))
    if cfg.channel == 'hierarchical':
        return Lead(L, N, cfg.K, cfg.lam, cfg.cUCB, cfg.H)
    raise ValueError(f'unknown channel {cfg.channel}')


def makeTeam(cfg, N, seed, channel):
    rngs = spawnRngs(seed + 1000, N)
    if cfg.agent == 'random':
        return [randomAgent(r) for r in rngs]
    if cfg.agent == 'circuit':
        if cfg.channel == 'hierarchical':
            raise ValueError('circuit agents cannot use hierarchical channel')
        return [CircuitCommMember(loadCircuitAgent(seed=int(r.integers(2**31))), channel, int(cfg.w)) for r in rngs]
    if cfg.channel == 'hierarchical':
        return [LeadFollower(channel) for _ in range(N)]
    return [CommBandit(channel, cfg.w, cfg.K, cfg.cUCB, cfg.H) for _ in range(N)]


def netHarvest(world, L):
    "Per ledger row: true harvest minus `dangerCost` x time spent at a danger location (welfare from Phase 7 on)."
    hit = np.isin(L['location'], sorted(world.danger)) & np.isfinite(L['s0'])
    return L['harvest'] - world.dangerCost * np.where(hit, L['s1'] - L['s0'], 0.0)


def harnessMetrics(cfg, world, hist, channel, members, N, T):
    L = ledgerArrays(world.ledger)
    gross, net = welfare(L, N, T)[0], netHarvest(world, L)
    tot = float(net.sum())
    q = L['t'] < T / 4
    firstQ = float(net[q].sum()) / (T / 4) if T > 0 else 0.0
    harvest_by_agent = [float(L['harvest'][L['agent'] == i].sum()) for i in range(N)]
    out = dict(config=cfg.asDict(), N=N, T=T, agentSeconds=N * T, welfare=tot, perAgentSecond=tot / (N * T),
               harvest=gross, dangerCost=gross - tot, firstQuarterRate=firstQ, gini=gini(np.array(harvest_by_agent)))
    if cfg.agent != 'circuit':
        times, counts = occupancy(L, N, world.nPatches, T)
        n_ifd, _ = idealFree(world, N)
        d = ifdDistanceTies(counts, n_ifd, world.intake, N)
        moves = [k * world.Tmove for k in range(1, int(T // world.Tmove) + 1) if k * world.Tmove < T] if world.Tmove else []
        _, y_so = socialOptimum(world, N)
        out.update(coverage=coverage(counts), dangerShare=dangerShare(L, world.danger, N, T),
                   ifdMean=float(d.mean()), timeToIFD=timeToIFD(times, d, moves),
                   soYield=y_so, welfareLoss=welfareLoss(tot, y_so, T))
    comm = dict(written=0, sent=0, received=0)
    for st in (channel.stats.values() if channel is not None else ()):
        for k in comm:
            comm[k] += st.get(k, 0)
    out['comm'] = {**comm, 'perAgentSecond': {k: v / (N * T) for k, v in comm.items()}}
    if cfg.agent == 'bandit' and cfg.channel != 'hierarchical':
        out['estimates'] = [dict(zip(('mean', 'n'), (m.tolist(), n.tolist())))
                            for m, n in [members[i].estimates(T) for i in range(N)]]
    elif cfg.agent == 'bandit' and cfg.channel == 'hierarchical' and channel is not None:
        S, C = channel.table.view(-1, T)
        out['leadView'] = dict(S=S.tolist(), C=C.tolist())
    elif cfg.agent == 'circuit':
        out['estimates'] = [{**goalStats(m.agent.policy.wG), 'a': int(m.agent.a), 'c': int(m.agent.c),
                             'nCirc': int(m.agent.params.nCirc)} for m in members]
    return out


def runHarness(cfg, N, seed):
    world = circuitWorld(**cfg.world) if cfg.agent == 'circuit' else benchWorld(**cfg.world)
    world.reset(seed, N)
    channel = makeChannel(cfg, world.L, N, seed)
    members = makeTeam(cfg, N, seed, channel)
    if cfg.agent == 'circuit':
        world.loc = [m.agent.a for m in members]
    T = worldTime(N, cfg)
    hist = runTeam(members, world, T)
    return world, hist, channel, harnessMetrics(cfg, world, hist, channel, members, N, T)


def _boot_ci(x, seed=0, n=2000):
    rng = np.random.default_rng(seed)
    x = np.asarray(x)
    b = [rng.choice(x, len(x), replace=True).mean() for _ in range(n)]
    return float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))


def rangeCheck(cfgs, sizes=(1, 4, 16, 64), seeds=range(10), T=600.0):
    """Phase 6 checks (1)-(3) on team harvest per second (whole run, first quarter), seed means for (1).
    All configs share the first one's world; needs a 'none' entry; table[str(N)][name] = [whole-run, first-quarter]."""
    channels, seeds, world = list(cfgs), list(seeds), dict(next(iter(cfgs.values())).world)

    def rates(cfg, N):
        c = HarnessConfig.fromDict({**cfg.asDict(), 'budget': 'time', 'T': T})
        ms = [runHarness(c, N, sd)[3] for sd in seeds]
        return np.array([[m['welfare'] / T, m['firstQuarterRate']] for m in ms])
    runs = {h: {N: rates(cfg, N) for N in sizes}
            for h, cfg in [*cfgs.items(), ('random', HarnessConfig(agent='random', world=world))]}
    so = {N: socialOptimum(benchWorld(**world), N)[1] for N in sizes}
    c1 = all(runs[h][N][:, 1].mean() < 0.9 * so[N] for h in channels for N in sizes)
    c2 = all(_boot_ci(runs['none'][N][:, 0])[0] > _boot_ci(runs['random'][N][:, 0])[1] for N in sizes)
    gaps = {}
    for N in sizes:
        means = {h: runs[h][N][:, 0].mean() for h in channels}
        best, worst = max(means, key=means.get), min(means, key=means.get)
        d = runs[best][N][:, 0] - runs[worst][N][:, 0]
        gaps[N] = (float(d.mean()),) + _boot_ci(d, seed=1)
    big, small = max(gaps, key=lambda N: gaps[N][0]), min(gaps, key=lambda N: gaps[N][0])
    c3 = sum(g[1] > 0 for g in gaps.values()) >= 2 and gaps[big][1] > gaps[small][2]
    table = {str(N): {h: runs[h][N].mean(0).tolist() for h in runs} for N in sizes}
    return dict(ceiling=c1, floor=c2, gap=c3, ok=bool(c1 and c2 and c3), gaps={N: list(g) for N, g in gaps.items()},
                table=table, so=so)

## Tests

In [ ]:
import time
import json
import tempfile
from pathlib import Path
import numpy as np
from fastcore.test import test_eq, test_close
from explRL.harness_runs import (HarnessConfig, benchWorld, circuitWorld, worldTime, makeChannel, makeTeam,
    runHarness, harnessMetrics, rangeCheck, netHarvest)
from explRL.multi import runTeam
from explRL.team_refs import ledgerArrays

# welfare is net of danger cost: one 1 s stay at a danger site (cost 2.6) and one at a patch (intake 1, c 0.85)
wd = benchWorld(nPatches=1, nDanger=1, intake=(1.0,), patchSites=[0], dangerSites=[1], noise=0.0)
wd.reset(0, 2); wd.loc = [1, 0]
wd.apply(0, 1, 0.0); wd.apply(1, 0, 0.0)
test_close(netHarvest(wd, ledgerArrays(wd.ledger)), [-2.6, 0.85])

# AC-8 budgets
cfg = HarnessConfig(budget='compute', B=15360.0)
test_close(worldTime(64, cfg), 240.0)
cfg2 = HarnessConfig(budget='time', T=600.0)
test_eq(worldTime(8, cfg2), 600.0)
w, h, ch, m = runHarness(HarnessConfig(channel='blackboard', budget='time', T=30.0), 4, 0)
test_close(m['agentSeconds'], 120.0)
assert all(r.t < 30.0 for r in h.records)
try:
    worldTime(0, cfg)
except ValueError:
    pass
else:
    raise AssertionError('expected ValueError')

# AC-9 determinism and config round-trip (travel rows hold nan start/end times, and nan != nan)
def same_ledger(a, b):
    test_eq(len(a), len(b))
    for ra, rb in zip(a, b):
        test_eq([x if x == x else 'nan' for x in ra], [x if x == x else 'nan' for x in rb])

cfgs = {k: HarnessConfig(channel=k) for k in ('none', 'blackboard', 'messages', 'hierarchical')}
for name, c in cfgs.items():
    w1, h1, ch1, m1 = runHarness(c, 4, 7)
    w2, h2, ch2, m2 = runHarness(c, 4, 7)
    same_ledger(w1.ledger, w2.ledger)
    test_eq(h1.records, h2.records)
    s1 = {k: dict(v) for k, v in (ch1.stats.items() if ch1 else [])}
    s2 = {k: dict(v) for k, v in (ch2.stats.items() if ch2 else [])}
    test_eq(s1, s2)
d = HarnessConfig(channel='messages', w=0.5, K=3).asDict()
test_eq(HarnessConfig.fromDict(d).asDict(), d)
with tempfile.TemporaryDirectory() as td:
    p = h1.toJSONL(Path(td) / 't.jsonl')
    assert p.exists()

# circuit team: deterministic too
for ch in ('none', 'blackboard', 'messages'):
    runs = [runHarness(HarnessConfig(agent='circuit', channel=ch, T=60.0), 2, 1) for _ in range(2)]
    same_ledger(runs[0][0].ledger, runs[1][0].ledger)
    test_eq(runs[0][1].records, runs[1][1].records)
    test_eq(len(runs[0][3]['estimates']), 2)
test_eq(set(m1['comm']), {'written', 'sent', 'received', 'perAgentSecond'})
test_close(m1['welfare'], m1['harvest'] - m1['dangerCost'])
test_eq(runHarness(HarnessConfig(), 2, 0)[3]['comm']['perAgentSecond']['written'], 0.0)

# rangeCheck runs end to end on a tiny grid, with the table layout the demo reads
rc = rangeCheck({k: HarnessConfig(channel=k) for k in ('none', 'blackboard')}, sizes=(1, 2), seeds=range(2), T=20.0)
test_eq(set(rc['table']['2']), {'none', 'blackboard', 'random'})
assert {'ceiling', 'floor', 'gap', 'ok'} <= set(rc)

# AC-10 speed: 64-agent blackboard team, 600 s
t0 = time.perf_counter()
_, hist, _, _ = runHarness(HarnessConfig(channel='blackboard'), 64, 0)
elapsed = time.perf_counter() - t0
per_event_us = elapsed / len(hist.records) * 1e6
print(f'AC-10: {elapsed:.2f} s for 64 agents x 600 s, {len(hist.records)} events, {per_event_us:.1f} us/event')
assert elapsed < 30.0, elapsed

# a torus benchmark world has 64 locations and runs; circuit estimates carry heading, compass and grid size
wt = benchWorld(shape='torus'); test_eq((wt.L, wt.layout.name), (64, 'torus'))
mt = runHarness(HarnessConfig(channel='blackboard', T=30.0, world=dict(shape='torus')), 4, 0)[3]
assert mt['welfare'] == mt['welfare']
ec = runHarness(HarnessConfig(agent='circuit', T=30.0), 2, 1)[3]['estimates'][0]
assert {'thetaG', 'a', 'c', 'nCirc'} <= set(ec)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()